# **Importing libraries and sample dataset**

In [ ]:
import warnings  # Import warnings module to control warning messages
warnings.filterwarnings('ignore')  # Suppress all warning messages during execution

In [ ]:
!pip install bnlearn==0.7.3  # Install bnlearn library for Bayesian network learning
!pip install numpy==1.19.5  # Install specific NumPy version for compatibility
!pip install pandas==1.3.5  # Install specific Pandas version for compatibility

In [ ]:
import bnlearn as bn  # Import bnlearn for Bayesian network and causal inference
import pandas as pd  # Import Pandas for data manipulation
import numpy as np  # Import NumPy for numerical operations

In [ ]:
pd.__version__,np.__version__  # Print installed Pandas and NumPy versions

('1.3.5', '1.21.5')

In [ ]:
df = bn.import_example('sprinkler')  # Load built-in sprinkler dataset from bnlearn
df.head(5)  # Display first 5 rows of the DataFrame

[bnlearn] >Import dataset..


,Cloudy,Sprinkler,Rain,Wet_Grass
0,0,0,0,0
1,1,0,1,1
2,0,1,0,1
3,1,1,1,1
4,1,1,1,1


# **Step 0: How different are our treatment and control groups?**


In [ ]:
df.groupby(['Sprinkler','Cloudy']).agg({'Wet_Grass':'count'})  # Count records per (Sprinkler, Cloudy) combination to check group imbalance

Wet_Grass
Sprinkler Cloudy           
0         0             225
          1             466
1         0             263
          1              46

We have a lot less cloudy days (46) on the group that has the treatment

# **Step 1: Calculating the propensity scores**

The Propensity Score is a conditional probability of being exposed given a set of covariates.

=

Probability of receiving a treatment (Sprinkler) given that it was cloudy


In [ ]:
#Calculating the probability of a cloudy day being part of the treatment

n_cloudy = len(df[df["Cloudy"]==1])  # Count total number of cloudy days in the dataset
n_sprinkler_cloudy = len(df[(df["Cloudy"]==1) & (df["Sprinkler"]==1)])  # Count cloudy days where sprinkler was also ON (treated + cloudy)
e_cloudy = n_sprinkler_cloudy/n_cloudy  # Compute propensity score for cloudy days: P(Sprinkler=1 | Cloudy=1)
print(e_cloudy)  # Print propensity score for cloudy days

0.08984375


In [ ]:
#Calculating the probability of a non cloudy day being part of the treatment

n_non_cloudy = len(df[df["Cloudy"]==0])  # Count total number of non-cloudy days in the dataset
n_sprinkler_non_cloudy = len(df[(df["Cloudy"]==0) & (df["Sprinkler"]==1)])  # Count non-cloudy days where sprinkler was ON (treated + not cloudy)
e_non_cloudy = n_sprinkler_non_cloudy/n_non_cloudy  # Compute propensity score for non-cloudy days: P(Sprinkler=1 | Cloudy=0)
print(e_non_cloudy)  # Print propensity score for non-cloudy days

0.5389344262295082


In [ ]:
df["propensity"] = np.where(df["Cloudy"]==1,e_cloudy,e_non_cloudy)  # Assign propensity score to each row based on its Cloudy value


In [ ]:
df  # Display the full DataFrame

,Cloudy,Sprinkler,Rain,Wet_Grass,propensity
0,0,0,0,0,0.538934
1,1,0,1,1,0.089844
2,0,1,0,1,0.538934
3,1,1,1,1,0.089844
4,1,1,1,1,0.089844
...,...,...,...,...,...
995,1,0,1,1,0.089844
996,1,0,1,1,0.089844
997,1,0,1,1,0.089844
998,0,0,0,0,0.538934


## **Step 2: Pairing each treated unit (sprinkler = 1), with a control unit with similar propensity**

In [ ]:
## Creating a dataframe only with 'treated' units
treated = df[df["Sprinkler"]==1]  # Filter dataset to keep only treated units (Sprinkler=ON)
treated = treated.reset_index(drop=True)  # Reset index of treated DataFrame for clean indexing
treated.head(5)  # Display first 5 rows of the DataFrame

,Cloudy,Sprinkler,Rain,Wet_Grass,propensity
0,0,1,0,1,0.538934
1,1,1,1,1,0.089844
2,1,1,1,1,0.089844
3,1,1,1,1,0.089844
4,0,1,0,1,0.538934


In [ ]:
## Creating a dataframe only with 'untreated' units
untreated = df[df["Sprinkler"]==0]  # Filter dataset to keep only untreated units (Sprinkler=OFF)

In [ ]:
## Function that adds to the treatment table a control unit with the same propensity score
matched_control = []  # Initialize empty list to collect matched control units
def add_matched_control(unit):  # Define function to find a matching control unit for each treated unit
    control_unit =untreated[untreated["propensity"]==unit["propensity"]].sample().iloc[0] ## Samples 1 unit of the untreated table  # Randomly sample one untreated unit with the same propensity score
    matched_control.append(control_unit) ## adds to the matched control list  # Add the matched control unit to the matched list

treated.apply(add_matched_control, axis=1)  ## runs function thrtough all the units in the treatment table  # Apply matching function to every treated unit row-by-row
matched_control_df = pd.DataFrame(matched_control).reset_index(drop=True)  ## creates a dataframe of matched controls  # Convert matched control list to DataFrame and reset index

In [ ]:
matched_control_df  # Display the matched control group DataFrame

,Cloudy,Sprinkler,Rain,Wet_Grass,propensity
0,0.0,0.0,1.0,0.0,0.538934
1,1.0,0.0,0.0,0.0,0.089844
2,1.0,0.0,1.0,1.0,0.089844
3,1.0,0.0,1.0,1.0,0.089844
4,0.0,0.0,1.0,1.0,0.538934
...,...,...,...,...,...
304,0.0,0.0,1.0,1.0,0.538934
305,1.0,0.0,1.0,1.0,0.089844
306,0.0,0.0,0.0,0.0,0.538934
307,1.0,0.0,1.0,1.0,0.089844


In [ ]:
treated  # Display the treated group DataFrame

,Cloudy,Sprinkler,Rain,Wet_Grass,propensity
0,0,1,0,1,0.538934
1,1,1,1,1,0.089844
2,1,1,1,1,0.089844
3,1,1,1,1,0.089844
4,0,1,0,1,0.538934
...,...,...,...,...,...
304,0,1,0,1,0.538934
305,1,1,1,1,0.089844
306,0,1,0,1,0.538934
307,1,1,0,1,0.089844


# **Step 3: Calculating the Average Treatment effect**

In [ ]:
##merging treatment and control functions
paired_sample = treated.join(matched_control_df, rsuffix="_control")  # Join treated and matched control DataFrames side-by-side

In [ ]:
paired_sample  # Display the paired sample with treated and control columns

,Cloudy,Sprinkler,Rain,Wet_Grass,propensity,Cloudy_control,Sprinkler_control,Rain_control,Wet_Grass_control,propensity_control
0,0,1,0,1,0.538934,0.0,0.0,1.0,0.0,0.538934
1,1,1,1,1,0.089844,1.0,0.0,0.0,0.0,0.089844
2,1,1,1,1,0.089844,1.0,0.0,1.0,1.0,0.089844
3,1,1,1,1,0.089844,1.0,0.0,1.0,1.0,0.089844
4,0,1,0,1,0.538934,0.0,0.0,1.0,1.0,0.538934
...,...,...,...,...,...,...,...,...,...,...
304,0,1,0,1,0.538934,0.0,0.0,1.0,1.0,0.538934
305,1,1,1,1,0.089844,1.0,0.0,1.0,1.0,0.089844
306,0,1,0,1,0.538934,0.0,0.0,0.0,0.0,0.538934
307,1,1,0,1,0.089844,1.0,0.0,1.0,1.0,0.089844


In [ ]:
ATE = (paired_sample["Wet_Grass"]-paired_sample["Wet_Grass_control"]).mean()  # Compute ATE: mean difference in Wet_Grass between treated and matched control
ATE  # Display the final Average Treatment Effect from propensity score matching

0.6245954692556634